# 01 — Training and Validation

## 1. Configuration

In [ ]:
import torch
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

# Paths (adjust to your Google Drive)
DATA_DIR  = Path('/content/drive/MyDrive/USK-Coffee')
LOCAL_DIR = Path('/content/USK-Coffee')
OUT_ROOT  = Path('/content/drive/MyDrive/coffee-two-stage/models')

# Data
CLASSES = ['defect', 'longberry', 'peaberry', 'premium']
EXPECTED_COUNT = {'train': 1200, 'val': 400, 'test': 400}
IMG_SIZE = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# Training (Syaputra & Rahardi, 2025)
BATCH_SIZE = 32
LR = 1e-4
STEP_SIZE, GAMMA = 5, 0.5
MAX_EPOCHS = 30
PATIENCE = 5
NUM_WORKERS = 2

# Experiment grid: 4class = flat, 3class = stage 1, 2class = stage 2
ARCHS   = ['resnet50', 'efficientnet_b0']
SCHEMES = ['4class', '3class', '2class']
SEEDS   = [0, 1, 2]

def run_dir(arch, scheme, seed):
    return OUT_ROOT / arch / scheme / f'seed{seed}'

## 2. Check image counts

In [ ]:
EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

def list_images(folder):
    return sorted(f for f in folder.iterdir() if f.suffix.lower() in EXTS)

for split, n_expected in EXPECTED_COUNT.items():
    for c in CLASSES:
        n = len(list_images(DATA_DIR / split / c))
        assert n == n_expected, f'{split}/{c}: {n} images, expected {n_expected}'
print('Image counts OK')

## 3. Copy data to the Colab disk

In [ ]:
import shutil, time

t0 = time.time()
for split in ['train', 'val']:
    dst = LOCAL_DIR / split
    ok = dst.exists() and all(
        (dst / c).exists() and len(list_images(dst / c)) == EXPECTED_COUNT[split]
        for c in CLASSES)
    if not ok:
        # Remove incomplete copy
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(DATA_DIR / split, dst)
    for c in CLASSES:
        assert len(list_images(dst / c)) == EXPECTED_COUNT[split], f'{split}/{c} incomplete'
print(f'done in {time.time() - t0:.0f}s')

## 4. Dataset and transforms

In [ ]:
import random
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

SCHEMES_MAP = {
    '4class': {'defect': 0, 'longberry': 1, 'peaberry': 2, 'premium': 3},
    '3class': {'defect': 0, 'longberry': 1, 'peaberry': 2, 'premium': 2},
    '2class': {'peaberry': 0, 'premium': 1},
}
LABEL_NAMES = {
    '4class': ['defect', 'longberry', 'peaberry', 'premium'],
    '3class': ['defect', 'longberry', 'peaberry+premium'],
    '2class': ['peaberry', 'premium'],
}

# Augmentation for training images only
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# 3class: peaberry and premium share one label
class CoffeeDataset(Dataset):
    def __init__(self, split_dir, scheme, transform):
        self.samples = [(f, label) for c, label in SCHEMES_MAP[scheme].items()
                        for f in list_images(split_dir / c)]
        self.transform = transform
        self.num_classes = len(LABEL_NAMES[scheme])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        with Image.open(path) as img:
            return self.transform(img.convert('RGB')), label

## 5. Training function

In [ ]:
import time
import pandas as pd
import torch.nn as nn
from torchvision import models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Reproducible runs per seed
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(_):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

def build_model(arch, num_classes, pretrained=True):
    if arch == 'resnet50':
        m = models.resnet50(weights='IMAGENET1K_V1' if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif arch == 'efficientnet_b0':
        m = models.efficientnet_b0(weights='IMAGENET1K_V1' if pretrained else None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    else:
        raise ValueError(arch)
    return m

@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs, labels = [], []
    for x, y in loader:
        probs.append(torch.softmax(model(x.to(device)), 1).cpu())
        labels.append(y)
    return torch.cat(probs).numpy(), torch.cat(labels).numpy()

def epochs_since_improvement(history):
    losses = [h['val_loss'] for h in history]
    return len(losses) - 1 - int(np.argmin(losses))

# Early stopping on validation loss
def should_stop(history):
    return len(history) >= MAX_EPOCHS or (len(history) > 0 and epochs_since_improvement(history) >= PATIENCE)

def train_model(arch, scheme, seed, out_dir, pretrained=True):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path  = out_dir / 'last_checkpoint.pt'
    best_path  = out_dir / 'best_model.pt'
    final_path = out_dir / 'final_model.pt'
    if final_path.exists():
        print(f'skip (done): {out_dir}')
        return

    set_seed(seed)
    train_ds = CoffeeDataset(LOCAL_DIR / 'train', scheme, train_tf)
    val_ds   = CoffeeDataset(LOCAL_DIR / 'val', scheme, eval_tf)
    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                              worker_init_fn=seed_worker, generator=g, pin_memory=True)
    val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    model = build_model(arch, train_ds.num_classes, pretrained).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=GAMMA)

    history = []
    # Resume after a Colab disconnect
    if ckpt_path.exists():
        ck = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        model.load_state_dict(ck['model']); optimizer.load_state_dict(ck['optimizer'])
        scheduler.load_state_dict(ck['scheduler']); history = ck['history']
        g.set_state(ck['rng_gen']); torch.set_rng_state(ck['rng_torch'])
        np.random.set_state(ck['rng_np']); random.setstate(ck['rng_py'])
        if ck['rng_cuda'] is not None: torch.cuda.set_rng_state_all(ck['rng_cuda'])
        print(f'resuming from epoch {len(history) + 1}')

    while not should_stop(history):
        epoch = len(history) + 1
        t0 = time.time()
        lr = optimizer.param_groups[0]['lr']
        model.train()
        loss_sum, correct, total = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(y)
            correct += (out.argmax(1) == y).sum().item(); total += len(y)
        scheduler.step()

        prob, y_val = predict(model, val_loader)
        val_loss = nn.functional.nll_loss(torch.log(torch.tensor(prob) + 1e-12), torch.tensor(y_val)).item()
        row = {'epoch': epoch, 'lr': lr, 'train_loss': loss_sum / total, 'train_acc': correct / total,
               'val_loss': val_loss, 'val_acc': float((prob.argmax(1) == y_val).mean()), 'sec': time.time() - t0}

        # Keep the weights with the lowest validation loss
        if row['val_loss'] < min((h['val_loss'] for h in history), default=float('inf')):
            torch.save(model.state_dict(), best_path)
        history.append(row)

        torch.save({'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                    'scheduler': scheduler.state_dict(), 'history': history,
                    'rng_gen': g.get_state(), 'rng_torch': torch.get_rng_state(),
                    'rng_np': np.random.get_state(), 'rng_py': random.getstate(),
                    'rng_cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None},
                   ckpt_path)
        pd.DataFrame(history).to_csv(out_dir / 'history.csv', index=False)
        print(f"  epoch {epoch:>2}/{MAX_EPOCHS} | train {row['train_loss']:.4f}/{row['train_acc']:.4f} "
              f"| val {val_loss:.4f}/{row['val_acc']:.4f} | {row['sec']:.0f}s")

    hist = pd.DataFrame(history)
    best = hist.loc[hist['val_loss'].idxmin()]
    best_epoch = int(best['epoch'])
    print(f"  stopped after epoch {len(hist)} | best val_loss {best['val_loss']:.4f} "
          f"(val_acc {best['val_acc']:.4f}) at epoch {best_epoch}")

    # Predict all validation images with the best weights
    model.load_state_dict(torch.load(best_path, map_location=device))
    full_val = CoffeeDataset(LOCAL_DIR / 'val', '4class', eval_tf)
    prob, y_true = predict(model, DataLoader(full_val, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS))
    np.savez(out_dir / 'val_predictions.npz', prob=prob, y_true=y_true,
             paths=np.array([str(p.relative_to(LOCAL_DIR)) for p, _ in full_val.samples]))

    # final_model.pt = lowest validation loss
    best_path.replace(final_path)
    ckpt_path.unlink(missing_ok=True)

## 6. Train the 18 models

In [ ]:
# Completed runs are skipped, interrupted runs resume
runs = [(seed, arch, scheme) for seed in SEEDS for arch in ARCHS for scheme in SCHEMES]

for i, (seed, arch, scheme) in enumerate(runs, 1):
    print(f'[{i:>2}/{len(runs)}] {arch} | {scheme} | seed {seed}')
    train_model(arch, scheme, seed, run_dir(arch, scheme, seed))
print('All runs finished')

## 7. Training status

In [ ]:
status = []
for seed, arch, scheme in runs:
    d = run_dir(arch, scheme, seed)
    row = {'arch': arch, 'scheme': scheme, 'seed': seed}
    if (d / 'history.csv').exists():
        h = pd.read_csv(d / 'history.csv')
        b = h.loc[h['val_loss'].idxmin()]
        row.update({'epochs_run': len(h), 'best_epoch': int(b['epoch']),
                    'best_val_loss': round(b['val_loss'], 4), 'val_acc_at_best': round(b['val_acc'], 4)})
    row['status'] = ('done' if (d / 'final_model.pt').exists()
                     else 'running/interrupted' if (d / 'history.csv').exists() else 'not started')
    status.append(row)
print(pd.DataFrame(status).to_string(index=False))

## 8. Validation results

In [ ]:
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

def load_preds(arch, scheme, seed):
    f = run_dir(arch, scheme, seed) / 'val_predictions.npz'
    return np.load(f) if f.exists() else None

# Merged-class images go to stage 2; others keep the stage-1 label
def two_stage_predict(s1_prob, s2_prob):
    # Stage 1: 0 defect, 1 longberry, 2 merged
    s1 = s1_prob.argmax(1)
    # Stage 2: 0 peaberry, 1 premium
    s2 = s2_prob.argmax(1)
    # Back to the four classes
    return np.where(s1 == 2, 2 + s2, s1)

def metrics_row(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=range(4), zero_division=0)
    errors = (y_true != y_pred).sum()
    pb_to_pr = ((y_true == 2) & (y_pred == 3)).sum()
    pr_to_pb = ((y_true == 3) & (y_pred == 2)).sum()
    row = {'accuracy': (y_true == y_pred).mean(), 'macro_f1': f1.mean(), 'errors': errors,
           'peaberry->premium': pb_to_pr, 'premium->peaberry': pr_to_pb,
           'pair_share_of_errors': (pb_to_pr + pr_to_pb) / max(errors, 1)}
    for k, c in enumerate(CLASSES):
        row.update({f'P_{c}': p[k], f'R_{c}': r[k], f'F1_{c}': f1[k]})
    return row

rows, cms = [], {}
for arch in ARCHS:
    for seed in SEEDS:
        base, s1, s2 = (load_preds(arch, sc, seed) for sc in SCHEMES)
        if base is not None:
            y = base['y_true']; pred = base['prob'].argmax(1)
            rows.append({'arch': arch, 'seed': seed, 'model': 'baseline', **metrics_row(y, pred)})
            cms[(arch, 'baseline', seed)] = confusion_matrix(y, pred, labels=range(4))
        if s1 is not None and s2 is not None:
            y = s1['y_true']; pred = two_stage_predict(s1['prob'], s2['prob'])
            s1_pred = s1['prob'].argmax(1)
            pair = np.isin(y, [2, 3])
            extra = {
                # Stage-1 accuracy
                'stage1_acc': (np.where(y >= 2, 2, y) == s1_pred).mean(),
                'stage2_acc_on_pair': (s2['prob'].argmax(1)[pair] == y[pair] - 2).mean(),
                # Pair images lost at stage 1
                'pair_lost_at_stage1': (pair & (s1_pred != 2)).sum(),
                'other_routed_to_stage2': (~pair & (s1_pred == 2)).sum(),
            }
            rows.append({'arch': arch, 'seed': seed, 'model': 'two_stage', **metrics_row(y, pred), **extra})
            cms[(arch, 'two_stage', seed)] = confusion_matrix(y, pred, labels=range(4))

results = pd.DataFrame(rows)
results.to_csv(OUT_ROOT / 'val_results_per_seed.csv', index=False)

main_cols = ['accuracy', 'macro_f1', 'R_peaberry', 'R_premium', 'peaberry->premium',
             'premium->peaberry', 'pair_share_of_errors']
summary = results.groupby(['arch', 'model'])[main_cols].agg(['mean', 'std'])
summary.to_csv(OUT_ROOT / 'val_results_summary.csv')

pd.set_option('display.width', 200)
print('Per seed:')
print(results[['arch', 'model', 'seed'] + main_cols].round(4).to_string(index=False))
print('\nMean ± std across seeds:')
print(summary.round(4).to_string())

stage_cols = ['stage1_acc', 'stage2_acc_on_pair', 'pair_lost_at_stage1', 'other_routed_to_stage2']
if 'stage1_acc' in results:
    print('\nTwo-stage details (error propagation):')
    print(results[results['model'] == 'two_stage'][['arch', 'seed'] + stage_cols].round(4).to_string(index=False))

## 9. Validation confusion matrices

In [ ]:
import matplotlib.pyplot as plt

pairs = [(a, m) for a in ARCHS for m in ['baseline', 'two_stage']
         if any((a, m, s) in cms for s in SEEDS)]
fig, axes = plt.subplots(1, len(pairs), figsize=(4.6 * len(pairs), 4.2), squeeze=False)
for ax, (arch, model_name) in zip(axes[0], pairs):
    mats = [cms[(arch, model_name, s)] for s in SEEDS if (arch, model_name, s) in cms]
    cm = sum(mats)
    ax.imshow(cm, cmap='Blues')
    for i in range(4):
        for j in range(4):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black')
    ax.set_xticks(range(4), CLASSES, rotation=45); ax.set_yticks(range(4), CLASSES)
    ax.set_xlabel('predicted'); ax.set_ylabel('true')
    ax.set_title(f'{arch}\n{model_name} ({len(mats)} seed)')
plt.tight_layout()
plt.savefig(OUT_ROOT / 'val_confusion_matrices.png', dpi=200)
plt.show()

## 10. Training histories

In [ ]:
histories = []
for seed, arch, scheme in runs:
    f = run_dir(arch, scheme, seed) / 'history.csv'
    if f.exists():
        h = pd.read_csv(f)
        h.insert(0, 'seed', seed); h.insert(0, 'scheme', scheme); h.insert(0, 'arch', arch)
        histories.append(h)
all_hist = pd.concat(histories)
all_hist.to_csv(OUT_ROOT / 'all_training_histories.csv', index=False)
print(all_hist.groupby(['arch', 'scheme', 'seed']).size().rename('epochs_run'))